# Week 2 — Which of these worlds could have liquid water?

[**Open this notebook in DataHub**](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs%2Fnotebooks%2F02_liquid_water.ipynb&branch=main)

## The question

Liquid water is the one thing every search for life beyond Earth agrees on looking for, and there
is a temperature range where water is liquid: between freezing and boiling. So here is a test you
can write in four lines. Work out how hot a planet's star makes it. If the answer lands between
273 K and 373 K, keep the planet. Otherwise throw it away.

NASA's Exoplanet Archive holds thousands of planets around other stars, and enough of them carry
the three numbers this test needs — the star's temperature, the star's size, and how far out the
planet orbits — that you can run it on all of them at once. You will.

Then you will point the same test at Earth, and it will throw Earth away. That failure is the
whole point of the week: it is not a bug in your arithmetic, and finding out what is missing tells
you more about this planet than a list of candidate worlds ever could.

## What you'll be able to do

**The Earth and planetary science**

- work out the temperature starlight alone would give any planet, from three published numbers;
- say what a planet's *albedo* is and what changing it does to that answer;
- measure the greenhouse effect, as the gap between the temperature a world should have and the
  one it does have — for Venus, Earth and Mars, three thicknesses of air;
- say why a list of "habitable" worlds built this way is not to be believed, and what it is
  missing.

**The Python**

- `for` loops, `range`, and `list.append` — doing the same thing to every item in a list;
- `if` / `elif` / `else` and the comparison operators, including what happens when the number you
  wanted to compare is not there;
- the accumulator pattern, for counting and collecting as a loop runs;
- `def`, arguments, `return`, and a docstring you can read back with `help()`.

## How this notebook works

A notebook is a stack of **cells**. Grey cells hold Python; white cells (like this one) hold text.
Click a cell and press **Shift + Enter** to run it. Everything runs inside a **kernel**, which
remembers what you have run so far. When things stop making sense, use
**Kernel ▸ Restart Kernel and Run All Cells**.

**Ten places where you write something: seven in class, three at home.** Nine of them are grey
cells, and nothing else in the notebook looks like this:

```python
# ← your answer here
```

The tenth wants a paragraph instead of code, and is a white cell reading *"Double-click this cell
and replace this line with your answer."*

The seven in class are six questions and one prediction. The three at home are the homework parts,
the last of which is the paragraph. All of it is your work and all of it is graded.

If you fall behind, look for a **Checkpoint** cell — running it rebuilds everything the next
section needs.

## Setup

Run this cell. You do not need to follow it yet.

**Coming later:** it uses **pandas** (week 3) to fetch a table off the web, and `def` (later
today) to give that job a name. What it hands you is seven plain lists of the kind you met last
week, all the same length and all in the same order.

**If this cell is still spinning after half a minute**, the archive has stalled. It does that
often, and it stalls silently rather than raising an error, so there is nothing to read. Press the
■ (stop) button in the toolbar to interrupt it, then run the cell again.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load():
    """Ask the NASA Exoplanet Archive for one row per known planet."""
    try:
        return pd.read_csv((
            "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query="
            "select+pl_name,st_teff,st_rad,pl_orbsmax,pl_rade,pl_eqt,discoverymethod"
            "+from+ps+where+default_flag=1&format=csv"))
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + "week02_exoplanets.csv")

SUN_TEMP = 5772.0             # the Sun's surface temperature in kelvin (IAU 2015 nominal value)
SUN_RADIUS_IN_AU = 0.00465047 # the Sun's radius, in astronomical units

archive = load()
usable = archive.dropna(subset=["st_teff", "st_rad", "pl_orbsmax"])

PLANETS_IN_ARCHIVE = len(archive)
names = list(usable.pl_name)
star_temps = list(usable.st_teff)                # the star's surface temperature, kelvin
star_radii = list(usable.st_rad)                 # the star's radius, in Suns
distances = list(usable.pl_orbsmax)              # how far the planet orbits out, in AU
radii = list(usable.pl_rade.astype(object).where(usable.pl_rade.notnull(), None))
archive_temps = list(usable.pl_eqt.astype(object).where(usable.pl_eqt.notnull(), None))
methods = list(usable.discoverymethod)         # how each planet was found

## 1. The temperature starlight alone would give a world

A planet has no worthwhile furnace of its own. It catches a disc of its star's light and it
radiates heat away from its whole sphere, and it settles at whatever temperature makes those two
equal. Write that balance down and the planet's own size cancels out of it — which is why a test
for liquid water needs nothing about the planet except how far out it orbits:

$$T \;=\; T_{\star}\,\sqrt{\frac{R_{\star}}{2d}}$$

$T_{\star}$ is the star's surface temperature, $R_{\star}$ is the star's radius and $d$ is the
planet's distance from it. The star's radius and the distance have to be in the same units, and
the archive gives us the first in Suns and the second in AU, so `SUN_RADIUS_IN_AU` from the setup
cell converts one to the other. In Python, `** 0.5` is a square root.

Two things this leaves out, and we will come back to both. It assumes the planet absorbs every
scrap of starlight that reaches it — real planets reflect some straight back. And it assumes the
planet has no air. So the number it gives has a name of its own. **Equilibrium temperature.** The
temperature a planet would sit at if starlight were the only thing heating it and it had no air.

It is a convention rather than a measurement: astronomers publish it for almost every planet found,
which is exactly what makes it comparable across thousands of worlds at once.

Let us do Venus, then Mars. Venus orbits at 0.723332 AU and Mars at 1.523679 AU (NASA Planetary
Fact Sheet).

In [ ]:
venus_temp = SUN_TEMP * (1.0 * SUN_RADIUS_IN_AU / (2 * 0.723332)) ** 0.5
print(round(venus_temp, 1), "K")

mars_temp = SUN_TEMP * (1.0 * SUN_RADIUS_IN_AU / (2 * 1.523679)) ** 0.5
print(round(mars_temp, 1), "K")

Those two blocks differ by one number. That is the signal to stop typing and start looping.

A **`for` loop** takes a list and runs the same indented lines once for each item in it:

```python
for distance in world_distances:
    ...one indented line...
    ...another...
```

The name after `for` is yours to choose; on each pass through the loop it holds the next item.
The indentation is not decoration — it is how Python knows which lines belong to the loop.

`list.append(x)` adds one item to the end of a list. Starting from an empty list `[]` and
appending inside a loop is how you build a result up one item at a time.

In [ ]:
world_names = ["Mercury", "Venus", "Mars", "Jupiter"]
world_distances = [0.387098, 0.723332, 1.523679, 5.204400]   # AU, NASA Planetary Fact Sheet

world_temps = []
for distance in world_distances:
    world_temps.append(SUN_TEMP * (1.0 * SUN_RADIUS_IN_AU / (2 * distance)) ** 0.5)

print(world_temps)

Four temperatures, and sixteen digits of noise on the end of each — `round()` from last week fixes
that. But the bigger problem is that they have arrived with no names attached, and reading them off
by counting is exactly the kind of thing you get wrong at 2 a.m.

`range(n)` gives you the whole numbers from 0 up to but not including `n`, so
`for i in range(len(world_names)):` walks `i` through every position in the list. Because
`world_names` and `world_temps` line up, position `i` in one describes the same world as position
`i` in the other.

### ✏️ Question 1

Print one line per world, reading from both lists, so that the first line reads

```
Mercury: 447.4 K
```

Use `range` and `len`, and round each temperature to one decimal place.

In [ ]:
# ← your answer here

## 2. Asking a question of every world at once

Water is liquid between **273 K** (freezing) and **373 K** (boiling) at Earth's sea-level air
pressure. Those two numbers are the whole test.

`if` runs its indented block only when a comparison is true, and the comparisons are the ones you
would write by hand: `<` `>` `<=` `>=` `==` (equal) and `!=` (not equal). `and` joins two of them
into one, and is true only when both halves are. There is `or` too, true when either half is.

The **accumulator pattern** is the other half of a loop. Set a counter to 0 before the loop starts
and add to it inside; same idea for a list — start it empty and append inside.

In [ ]:
too_cold = []
for i in range(len(world_names)):
    if world_temps[i] < 273:
        too_cold.append(world_names[i])

print(len(too_cold), "of the four are below freezing:", too_cold)

### ✏️ Question 2

Now the test itself. Build a list called `liquid_water_worlds` holding the names of the worlds
whose temperature is **between 273 and 373 K inclusive** — one `if` with an `and` in it — and
print how many there are and which.

**Use that name**, `liquid_water_worlds`, because the sentence below reads it back.

In [ ]:
# ← your answer here

One world out of four, and it is Venus. Hold on to that; section 7 comes back to it with a
thermometer.

You may also have noticed which world is missing from `world_names`. That was deliberate. Earth
comes back in section 6, and by then you will have built the test properly.

## 3. Three thousand worlds

Four planets do not tell you whether the test is any good. The setup cell fetched NASA's Exoplanet
Archive, which holds one row per known planet around another star, and kept every planet carrying
all three numbers the formula needs.

Last week's idea applies here too, word for word: **a catalogue lists what somebody's instruments
recorded, not what happened.** A planet is in this file because a telescope could detect it, and
the numbers it carries are the ones that particular detection method can measure.

In [ ]:
print(PLANETS_IN_ARCHIVE, "planets in the archive")
print(len(names), "of them carry a star temperature, a star radius and an orbital distance")
print(names[0], star_temps[0], "K,", star_radii[0], "Suns,", distances[0], "AU")

by_transit = 0
for method in methods:
    if method == "Transit":
        by_transit = by_transit + 1
print(by_transit, "of the", len(names), "were found by watching their star dim as they crossed it")

The loop you wrote for four worlds is the same loop for three thousand. That is the whole reason
loops exist: the number of items stops being your problem.

In [ ]:
temps = []
for i in range(len(names)):
    temps.append(star_temps[i] * (star_radii[i] * SUN_RADIUS_IN_AU / (2 * distances[i])) ** 0.5)

hotter_than_1500 = 0
for temperature in temps:
    if temperature > 1500:
        hotter_than_1500 = hotter_than_1500 + 1

print(len(temps), "temperatures, from", round(min(temps), 1), "K to", round(max(temps), 1), "K")
print(hotter_than_1500, "of them are above 1500 K")

In [ ]:
plt.hist(temps, bins=range(0, 1550, 50))
plt.axvline(273, color="C1")        # water freezes
plt.axvline(373, color="C1")        # water boils
plt.xlabel("equilibrium temperature (K)")
plt.ylabel("number of planets")
plt.title(f"{len(temps)} planets; the {hotter_than_1500} above 1500 K are off the right edge")
plt.show()

`plt.axvline(x)` draws a vertical line at `x` — here the two edges of the window. They are only
100 K apart while the temperatures run past 1500 K, so the window is a thin slice of what is out
there, and the bulk of the distribution sits to the right of it, hotter. That is mostly a fact
about the telescopes: three quarters of these planets were found by watching a star dim as the
planet crossed in front of it, a planet close in crosses more often and lines up more often, and
close in means hot.

Before trusting our own arithmetic, check it against somebody else's. The archive publishes its own
equilibrium temperature for some of these planets. `abs(x)` gives the size of a number ignoring its
sign, which is how you ask "how far apart are these two" without caring which is bigger.

In [ ]:
ours = []
published = []
for i in range(len(names)):
    if archive_temps[i] is not None:
        ours.append(temps[i])
        published.append(archive_temps[i])

close = 0
for i in range(len(ours)):
    if abs(ours[i] - published[i]) <= 10:
        close = close + 1

print(len(ours), "planets have a published equilibrium temperature to compare against")
print(close, "of those agree with ours to within 10 K —", round(100 * close / len(ours)), "percent")

In [ ]:
plt.scatter(published, ours, s=4, alpha=0.3)
plt.plot([0, 4000], [0, 4000], color="0.4", lw=1)   # the line where the two would agree exactly
plt.xlabel("the archive's published equilibrium temperature (K)")
plt.ylabel("our equilibrium temperature (K)")
plt.title(f"our arithmetic against the archive's, {len(ours)} planets")
plt.show()

The cloud lies along the grey line and three in five agree to within 10 K, which is enough to say
our formula is the standard formula. It is not enough to say the two always agree. The archive's
number comes from whichever paper reported that planet, and papers differ in the albedo they assume
and the orbital distance they adopt, so where the two part company nothing in this file tells you
which of them is right. Look at the handful of points along the bottom of the plot: our formula
returns almost nothing for a planet the archive calls hot, which means the distance in that row and
the temperature beside it cannot both be describing the same orbit.

### ✏️ Question 3

Run the liquid-water test on the whole archive. Count how many of the temperatures in `temps` are
between 273 and 373 K inclusive, and print the count and how many planets you tested.

**Use the name** `in_window` for the count.

In [ ]:
# ← your answer here

## 4. Rocky, gassy, or nobody knows

A planet the size of Neptune has no surface for an ocean to sit on. The usual dividing line is
**1.6 Earth radii**: below it planets are almost all dense enough to be rock, above it almost all
have thick hydrogen envelopes. That is a convention drawn from measured masses and radii, not a
law, and planets near the line go either way.

The archive has `pl_rade`, the planet's radius in Earths — but not for every planet. Some planets
were found by a method that cannot measure a radius at all. Where the number is missing, `load()`
put **`None`** there, which is Python's word for "nothing here". You cannot compare `None` with a
number: `None < 1.6` is an error, not an answer. So the test needs three branches rather than two,
and `if` / `elif` / `else` is how you write them: Python tries each in order and runs the first one
that is true.

In [ ]:
rocky_worlds = []
too_big = 0
unknown_radius = 0
for i in range(len(names)):
    if temps[i] >= 273 and temps[i] <= 373:
        if radii[i] is None:
            unknown_radius = unknown_radius + 1
        elif radii[i] < 1.6:
            rocky_worlds.append(names[i])
        else:
            too_big = too_big + 1

print(len(rocky_worlds), "rocky,", too_big, "too big,", unknown_radius, "with no measured radius")
print("adding up to", len(rocky_worlds) + too_big + unknown_radius, "planets in the window")
print(rocky_worlds)

In [ ]:
for planet in ["TRAPPIST-1 c", "TRAPPIST-1 d", "TRAPPIST-1 e", "TRAPPIST-1 f"]:
    i = names.index(planet)
    print(planet, round(temps[i], 1), "K")

Look at the names on the list, and then at those four planets of the TRAPPIST-1 system, which all
orbit the same star at increasing distances. The test keeps c and d and throws out e and f — and
that is the wrong way round from how the exoplanet literature reads that system, where e is the
planet most often named as its best candidate for liquid water and c is usually discussed as a
likely Venus analogue. Your test has kept the one people doubt and rejected the one people like,
and the arithmetic that did it is not wrong.

So the test accepts twelve worlds — and the third branch is the interesting one. 102 of the 189
planets in the window have no measured radius at all, more than half of them, and any of those
could be rocky. Twelve is a floor on the answer, not the answer. **A catalogue lists what
somebody's instruments recorded, not what happened**, and here what was not recorded is most of
the evidence.

Keep `rocky_worlds`. The homework compares its own list against it.

## 5. Writing the question down once

In [ ]:
# ── Checkpoint ── run this if you are behind ──
temps = []
for i in range(len(names)):
    temps.append(star_temps[i] * (star_radii[i] * SUN_RADIUS_IN_AU / (2 * distances[i])) ** 0.5)

rocky_worlds = []
for i in range(len(names)):
    if temps[i] >= 273 and temps[i] <= 373 and radii[i] is not None and radii[i] < 1.6:
        rocky_worlds.append(names[i])

We are about to ask the same question with a different number in it, several times over. Copying
the loop and editing one value is what you would do next, and it is exactly what goes wrong: five
copies of a formula are five chances to mistype it and no way to tell which copy is the one that
is right.

A **function** is a recipe written once and given a name. `def` starts it, the names in brackets
are what it needs, the indented lines are what it does, and `return` hands one value back. The
first thing inside should be a **docstring**: one line in triple quotes saying what the function is
for. `help()` prints it back at you, which is why writing one pays.

In [ ]:
def to_celsius(kelvin):
    """Convert a temperature in kelvin to degrees Celsius."""
    return kelvin - 273.15


print(round(to_celsius(288.0), 2))
help(to_celsius)

Now the real one — and this is where the missing physics goes in.

**Albedo.** The fraction of the starlight falling on a world that it reflects straight back into
space; only the rest is absorbed and turned into heat. Everything so far assumed an albedo of 0, a
perfectly black world. Putting it in multiplies the temperature by $(1-A)^{1/4}$:

$$T \;=\; T_{\star}\,(1-A)^{1/4}\,\sqrt{\frac{R_{\star}}{2d}}$$

### ✏️ Question 4

Write that formula down once, as a function:

```python
def equilibrium_temperature(star_temp, star_radius, distance, albedo):
```

It returns `star_temp` × `(1 - albedo) ** 0.25` × the square root of
`star_radius * SUN_RADIUS_IN_AU / (2 * distance)`. Give it a one-line docstring.

Then check it against something you already know: call it for Mars — `SUN_TEMP`, a star radius of
`1.0`, a distance of `1.523679` and an albedo of `0.0` — and print the answer rounded to one
decimal. You should get back the number section 1 printed for Mars.

**Use this name**, `equilibrium_temperature`, because every cell below calls it.

In [ ]:
# ← your answer here

## 6. Now point it at Earth

Two worlds have been missing from every list in this notebook. Earth is one of them; Venus is the
other, and Venus is the only solar-system world your test has accepted so far.

Both have had their albedo measured from orbit: **Earth 0.306**, **Venus 0.770** (NASA Planetary
Fact Sheet). Earth orbits at 1.000 AU and Venus at 0.723332 AU.

Earth's average surface temperature is 288 K — 15 °C, the number you have known since school.
Before you run anything, commit to a guess: set `guess_earth` to the temperature in kelvin you
think `equilibrium_temperature` will hand back for Earth.

In [ ]:
# ← your answer here

### ✏️ Question 5

Call `equilibrium_temperature` four times, and print each answer rounded to one decimal along with
whether it lands inside the 273–373 K window:

- Earth with albedo `0.0`, then Earth with its measured `0.306`;
- Venus with albedo `0.0`, then Venus with its measured `0.770`.

In [ ]:
# ← your answer here

Read those four lines again, because between them they demolish the test you just built.

Assume both worlds are perfectly black and the test **accepts both** — Venus at 327.3 K and Earth
at 278.3 K, five degrees above freezing. Use the albedos somebody actually measured and it
**rejects both** — Venus at 226.6 K and Earth at 254.0 K, nineteen degrees below freezing.

There is no third run where it gets the answer right. One of these two worlds has oceans and the
other does not, and the test never separates them: it puts them 49 K apart at albedo 0 and 27 K
apart at their measured albedos, either way with Venus the cooler of the two. Which way it is
wrong turns on the albedo, a number that says nothing about whether anybody could live there. The
next section measures how large the thing it is missing actually is.

## 7. The size of what is missing

You have a number for the temperature a world should have. For the worlds next door somebody has
been and measured the temperature they do have. The difference is not an error term — it is a
quantity, and one you can now put a size on.

The mean surface temperatures and surface pressures below are measured values from the NASA
Planetary Fact Sheet; the albedos are the Bond albedos from the same source. The plot after them
uses one new drawing command, `plt.text(x, y, "Venus")`, which writes a word at a point on the
axes — with three dots and no labels the figure would say nothing.

In [ ]:
solar_names = ["Venus", "Earth", "Mars"]
solar_distances = [0.723332, 1.000000, 1.523679]   # AU
solar_albedos = [0.770, 0.306, 0.250]              # Bond albedo
surface_temps = [737.0, 288.0, 210.0]              # measured mean surface temperature, K
pressures = [92.0, 1.014, 0.006]                   # surface pressure, bar

no_air_temps = []
for i in range(len(solar_names)):
    no_air = equilibrium_temperature(SUN_TEMP, 1.0, solar_distances[i], solar_albedos[i])
    no_air_temps.append(no_air)
    print(f"{solar_names[i]:6s} starlight alone {no_air:6.1f} K   measured {surface_temps[i]:6.1f} K"
          f"   gap {surface_temps[i] - no_air:+7.1f} K   air {pressures[i]:6.3f} bar")

In [ ]:
plt.scatter(no_air_temps, surface_temps)
plt.plot([150, 800], [150, 800], color="0.4", lw=1)   # a world with no air would sit on this line
for i in range(len(solar_names)):
    plt.text(no_air_temps[i] + 15, surface_temps[i], solar_names[i])
plt.xlim(150, 800)                     # the same range on both axes, so the grey line is a fair
plt.ylim(150, 800)                     # comparison rather than an accident of scaling
plt.gca().set_aspect("equal")
plt.xlabel("equilibrium temperature, using each world's measured albedo (K)")
plt.ylabel("measured mean surface temperature (K)")
plt.title("what the air adds: 3 worlds")
plt.show()

All three should be on the grey line. All three are above it, and the plot makes the reason
visible: along the bottom axis the three worlds sit between 209.8 K and 254.0 K, inside forty-five
degrees of each other, while up the side they run from 210.0 K to 737.0 K. Whatever separates
Venus from Mars, the starlight arithmetic cannot see it.

The gap has a name. **The greenhouse effect is the difference between the temperature a world
would have with no air and the temperature it actually has** — the atmosphere lets sunlight down
and slows the infrared going back out. It is not a definition we imposed here; it is what is left
over after the arithmetic, and it comes out at +34.0 K for Earth under 1.014 bar of air, +510.4 K
for Venus under 92 bar, and +0.2 K for Mars under 0.006 bar. Mars's mean surface temperature is
quoted to the nearest degree, so +0.2 K is indistinguishable from nothing at all: three
thicknesses of air, and the more air a world carries the more it warms — but not in proportion.
Venus has ninety times Earth's surface pressure and fifteen times its warming, so what the air is
made of matters as much as how much of it there is.

That is why the test threw Earth away. It was never a test of habitability. It was a test of how
much starlight arrives, and the 34 K that makes this planet habitable is not in it.

## 8. Asking again, and again

The window edges were a choice too. 273 K and 373 K are water's freezing and boiling points at
Earth's sea-level pressure — a planet with thinner air boils water lower, and a planet with more
holds it liquid higher. Nobody has measured the air pressure on any of these worlds.

So we should ask the question several ways. That is what the function was for: `count_worlds`
below is the whole of sections 3 and 4 written down once, and now the window and the albedo are
things you hand it rather than things you retype.

In [ ]:
def count_worlds(low, high, albedo):
    """How many archive planets with a measured radius under 1.6 Earths sit between low and high K."""
    found = 0
    for i in range(len(names)):
        temperature = equilibrium_temperature(star_temps[i], star_radii[i], distances[i], albedo)
        if radii[i] is not None and radii[i] < 1.6 and temperature >= low and temperature <= high:
            found = found + 1
    return found


print(count_worlds(180, 273, 0.0), "rocky worlds are below freezing but above 180 K")

### ✏️ Question 6

Call `count_worlds` three times, all with albedo `0.0`, and print the count each window gives:

- **273 to 373 K** — water's liquid range at Earth's sea-level pressure;
- **250 to 350 K** — the same width, shifted down, on the grounds that any atmosphere at all warms
  a planet above its equilibrium temperature;
- **273 to 323 K** — freezing to 50 °C, if you want somewhere merely uncomfortable.

Print the window and its count on each line. The first should hand back the twelve worlds section
4 found — that is your check that `count_worlds` is doing what the loops did.

In [ ]:
# ← your answer here

Three defensible windows, three different answers, from the same data and the same code. Nothing
in the archive decides between them — you decide, and then you report which one you chose. The
homework hands you the other choice, the albedo, and that one moves the answer much further.

## The question, answered

**Which of these worlds could have liquid water — and why does your test reject Earth?** On the
archive's own numbers the test accepts twelve rocky worlds, with a hundred and two more in the
window whose size nobody has measured; but it rejects Earth, because equilibrium temperature
counts the starlight arriving and nothing else, and the 34.0 K that keeps this planet's oceans
liquid comes from its air. Run the same test on Venus and Earth with an albedo of 0 and it accepts
both; run it with the albedos we have measured and it rejects both. It never gets the answer right,
and the size of how wrong it is — 34 K here, 510 K on Venus, nothing on Mars — is the greenhouse
effect, measured.

## Week 2 summary

**The question.** Which of these worlds could have liquid water — and why does your test reject Earth?

### What to remember

| | |
|---|---|
| **1** | Equilibrium temperature ignores atmospheres: run the test one way and it accepts Venus alongside Earth, run it the other and it rejects both. It never gets the answer right. |
| **2** | The greenhouse effect arrives as a measured discrepancy, not a definition. |
| **3** | A function is a question you can ask many times without retyping it. |

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Accumulator pattern** | Set a counter to 0 before the loop starts and add to it inside; same idea for a list — start it empty and append inside. |
| **Equilibrium temperature** | The temperature a planet would sit at if starlight were the only thing heating it and it had no air. |
| **Albedo** | The fraction of the starlight falling on a world that it reflects straight back into space; only the rest is absorbed and turned into heat. |
| **The greenhouse effect** | The difference between the temperature a world would have with no air and the temperature it actually has. |

### Code you met this week

| Function | What it does |
|---|---|
| `for x in things:` | do the same thing once for each item |
| `range(n)` | the numbers 0 to n-1, to count with |
| `list.append(x)` | add one item to the end of a list |
| `if / elif / else` | choose what to do, based on a comparison |
| `and / or / not` | combine two conditions into one |
| `abs(x)` | the size of a number, ignoring its sign |
| `x in things` | true when that value is somewhere in the list |
| `None` | Python's word for "nothing here" — you cannot compare it with a number |
| `plt.axvline(x)` | a vertical line at x, to mark a boundary on a plot |
| `plt.text(x, y, "Venus")` | write a word at a point on the axes |
| `def name(a, b):` | write the recipe once and give it a name; the names in brackets are what it needs |
| `return value` | hand one value back to whoever called the function |
| `a docstring` | triple-quoted text as the first thing inside a function, saying what it does |
| `return a, b` | hand back two values at once; catch them with a, b = f(...) |
| `help(f)` | print a function's docstring, which is why writing one pays |

## Homework

Three parts. Class ran the test at one albedo and asked which worlds passed; these ask where the
window actually is, what happens when you pick a different albedo, and whether you believe the
answer.

Each of the first two ends with a **self-check** cell: run it, and it tells you if something is
missing, then prints your own numbers back. (Those cells use `assert`, which stops with a message
when what follows it is false. You run them; you never have to write them.)

### ✏️ Homework 1 — where is the window?

Class asked *is this planet in the window?* Ask the other question: *where is the window?* For a
given star there are two distances, one where a planet would be 373 K and one where it would be
273 K, and every planet between them passes the test.

Turn the formula round. If

$$T \;=\; T_{\star}(1-A)^{1/4}\sqrt{\frac{R_{\star}}{2d}} \qquad\text{then}\qquad
d \;=\; \frac{R_{\star}}{2\,(T / T_{\text{eff}})^{2}} \quad\text{where}\quad
T_{\text{eff}} = T_{\star}(1-A)^{1/4}$$

Write it as a function:

```python
def window_edges(star_temp, star_radius, albedo):
```

returning **two** numbers — the inner edge (where a planet would be 373 K) and the outer edge
(where it would be 273 K), both in AU. `return inner, outer` hands back two values at once, and
you catch them with `inner, outer = window_edges(...)`, the same way the setup cell caught seven
lists from `load()`. Remember `SUN_RADIUS_IN_AU`, since the archive measures star radii in Suns
and distances in AU.

Then call it twice for the Sun — `SUN_TEMP`, a radius of `1.0` — once with albedo `0.0` and once
with Earth's measured `0.306`, and print both windows. Earth orbits at 1.000 AU: say for each
window whether Earth is inside it.

**Use these names**, because the self-check looks for them: `sun_inner_0`, `sun_outer_0`,
`sun_inner_earth`, `sun_outer_earth`.

In [ ]:
# ← your answer here

In [ ]:
assert sun_inner_0 < sun_outer_0, "the 373 K edge is the inner one — it is closer to the star"
assert sun_outer_earth < sun_outer_0, "a planet that reflects light is colder, so its window sits closer in"
print(f"With no reflection the Sun's window runs {sun_inner_0:.3f} to {sun_outer_0:.3f} AU.")
print(f"With Earth's albedo it runs {sun_inner_earth:.3f} to {sun_outer_earth:.3f} AU.")
print("Earth orbits at 1.000 AU.")

### ✏️ Homework 2 — the fork

Nobody has measured the albedo of any planet in the archive. Class used 0, a perfectly black
world, which is the one value we know is wrong. Two defensible substitutes, both measured, both in
this notebook:

- **Earth's, 0.306** — use the one rocky world we know has liquid water;
- **Venus's, 0.770** — most of the worlds this test accepts orbit close in and hot, and Venus is
  the rocky world we have that ended up that way; a thick, bright atmosphere is at least as
  ordinary an outcome as a thin, clear one.

Pick one. Set `my_albedo` to it, then rebuild the accepted list at that albedo: loop over the
archive, and for every planet whose radius is known and under 1.6 and whose temperature is between
273 and 373 K inclusive, append the name to `my_worlds` and the temperature to `my_temps`.

Then compare against class's list. `if name in rocky_worlds:` is true when that name is somewhere
in the list. Count into `stayed` the worlds on your list that were on class's twelve, and into
`moved_in` the ones that were not.

**Use these names**: `my_albedo`, `my_worlds`, `my_temps`, `stayed`, `moved_in`.

In [ ]:
# ← your answer here

In [ ]:
assert my_albedo == 0.306 or my_albedo == 0.770, "pick Earth's albedo or Venus's — those are the two anyone has measured"
assert len(my_worlds) == len(my_temps), "one temperature per world; append to both lists in the same pass"
assert len(my_worlds) > len(rocky_worlds), "a higher albedo cools every planet, so worlds that were too hot for class's window drop into yours — expect more than twelve"
assert stayed + moved_in == len(my_worlds), "every world on your list either was on class's list or was not"
print(f"At albedo {my_albedo} the test accepts {len(my_worlds)} worlds: {stayed} of class's "
      f"{len(rocky_worlds)} survived and {moved_in} are new.")
print("The first ten, with the numbers your test used on them:")
for i in range(10):
    j = names.index(my_worlds[i])
    print(f"  {my_worlds[i]:16s} {my_temps[i]:6.1f} K   star {star_temps[j]:6.0f} K"
          f"   orbit {distances[j]:.3f} AU")

### ✏️ Homework 3 — one you do not believe

Your test now calls a list of worlds habitable. Pick one of them you do not believe, and say what
the test is missing.

Three or four sentences, and make them specific to numbers you produced:

- name the world, and quote the equilibrium temperature your own run gave it;
- say what your test measured about that world and what it did not;
- class measured Earth's air adding 34.0 K and Venus's adding 510.4 K, to worlds whose starlight
  temperatures were 254.0 K and 226.6 K. Use those numbers to say how far wrong your figure for
  your world could be, in both directions.

Do not stop at "it ignores the atmosphere". Say what that world's atmosphere would have to be
doing for it to be habitable, and what it would have to be doing for it not to be.

*(Double-click this cell and replace this line with your answer.)*